# Profile Headers Analytics

This notebook helps you:

1. Run `ci/profile_headers.sh` from the repo root.
2. Load and inspect the generated CSV.
3. Explore expensive headers by compile time, transitive LOC, and include count.
4. Build combined ranking scores to identify high-impact headers.

Expected CSV columns:
- `header_path`
- `compile_time_ms`
- `transitive_loc`
- `included_by_header_count`

In [ ]:
%pip install pandas matplotlib plotly nbformat

In [ ]:
import os
import shlex
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Could not locate repo root (.git). Start notebook from inside the CCCL repo."
    )

print(f"Repo root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")

In [ ]:
# --- Run profile_headers.sh ---
import subprocess
from pathlib import Path

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
        REPO_ROOT = REPO_ROOT.parent

output_csv = Path("/tmp/profile_headers.csv")
cmd = [
    "bash",
    str(REPO_ROOT / "ci" / "profile_headers.sh"),
    "--output-csv",
    str(output_csv),
]

print("Command:")
print("  " + " ".join(shlex.quote(x) for x in cmd))

subprocess.run(cmd, cwd=REPO_ROOT, check=True)
print(f"\nWrote: {output_csv}")

In [ ]:
# --- Load CSV ---
csv_path = Path("/tmp/profile_headers.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Missing CSV: {csv_path}. Run the script first.")

df = pd.read_csv(csv_path)

# Defensive numeric conversion for robust downstream analysis.
for col in ["compile_time_ms", "transitive_loc", "included_by_header_count"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(5)

In [ ]:
# --- Quick top-N views ---
TOP_N = 5

print("Top by compile_time_ms")
display(
    df.nlargest(TOP_N, "compile_time_ms")[
        ["header_path", "compile_time_ms", "transitive_loc", "included_by_header_count"]
    ]
)

print("\nTop by transitive_loc")
display(
    df.nlargest(TOP_N, "transitive_loc")[
        ["header_path", "transitive_loc", "compile_time_ms", "included_by_header_count"]
    ]
)

print("\nTop by included_by_header_count")
display(
    df.nlargest(TOP_N, "included_by_header_count")[
        ["header_path", "included_by_header_count", "compile_time_ms", "transitive_loc"]
    ]
)

In [ ]:
# --- Interactive scatter (hover shows header name) ---

import plotly.express as px

plot_df = df.copy()

fig = px.scatter(
    plot_df,
    x="included_by_header_count",
    y="compile_time_ms",
    hover_name="header_path",
    hover_data={
        "transitive_loc": True,
        "included_by_header_count": True,
        "compile_time_ms": ":.2f",
        "header_path": False,
    },
    opacity=0.6,
    title="Include count vs compile time (hover for header)",
)

fig.update_layout(
    xaxis_title="included_by_header_count",
    yaxis_title="compile_time_ms",
    height=900,
)
fig.show()

In [ ]:
# --- Optional: run ctadvisor and parse expensive headers ---
CTADVISOR_ENTRIES = 10
CTADVISOR_THREADS = os.cpu_count() or 8

trace_root = (
    REPO_ROOT
    / "build"
    / os.environ.get("CCCL_BUILD_INFIX", "cuda13.1-gcc14")
    / "profile-headers"
    / "header_testing"
    / "device_time_trace"
)
ctadvisor_cmd = [
    "ctadvisor",
    "--trace-file-path",
    str(trace_root),
    "--header-advisor-entries",
    str(CTADVISOR_ENTRIES),
    "--thread-number",
    str(CTADVISOR_THREADS),
]

print("Command:")
print("  " + " ".join(shlex.quote(x) for x in ctadvisor_cmd))

result = subprocess.run(
    ctadvisor_cmd, cwd=REPO_ROOT, check=True, capture_output=True, text=True
)
print(result.stdout)